In [18]:
import pandas as pd

data = pd.read_csv(r'D:\My_Space\College\6th_SEM\Deep_Learning_Project\IDS\Data\processed\cicids2017_merged.csv')

<h2>STEP 1 – Clean Weird Label Encoding</h2>

In [19]:
data['Label'] = data['Label'].str.replace('�', '-', regex=False)

<h2>STEP 2 – Group Classes</h2>

In [20]:
def group_labels(label):
    
    if label == 'BENIGN':
        return 'BENIGN'
    
    elif 'DoS' in label and 'DDoS' not in label:
        return 'DoS'
    
    elif 'DDoS' in label:
        return 'DDoS'
    
    elif 'PortScan' in label:
        return 'PortScan'
    
    elif 'Patator' in label:
        return 'BruteForce'
    
    elif 'Web Attack' in label:
        return 'WebAttack'
    
    elif 'Bot' in label:
        return 'Bot'
    
    elif 'Infiltration' in label:
        return 'Infiltration'
    
    elif 'Heartbleed' in label:
        return 'Heartbleed'
    
    else:
        return 'Other'


data['Attack_Category'] = data['Label'].apply(group_labels)

In [21]:
data['Attack_Category'].value_counts()

Attack_Category
BENIGN          2095057
DoS              193745
DDoS             128014
PortScan          90694
BruteForce         9150
WebAttack          2143
Bot                1948
Infiltration         36
Heartbleed           11
Name: count, dtype: int64

<h2>Step 3 – Remove Rare Classes</h2>

In [22]:
data = data[~data['Attack_Category'].isin(['Infiltration', 'Heartbleed'])]

print(data['Attack_Category'].value_counts())

Attack_Category
BENIGN        2095057
DoS            193745
DDoS           128014
PortScan        90694
BruteForce       9150
WebAttack        2143
Bot              1948
Name: count, dtype: int64


<h2>STEP 4 – Encode Multi-Class Labels</h2>

In [23]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
data['Encoded_Label'] = label_encoder.fit_transform(data['Attack_Category'])

print("Classes:")
print(label_encoder.classes_)

Classes:
['BENIGN' 'Bot' 'BruteForce' 'DDoS' 'DoS' 'PortScan' 'WebAttack']


In [28]:
import os
import joblib

os.makedirs(r"D:\My_Space\College\6th_SEM\Deep_Learning_Project\IDS\models", exist_ok=True)

joblib.dump(label_encoder, r"D:\My_Space\College\6th_SEM\Deep_Learning_Project\IDS\models\label_encoder.pkl")

['D:\\My_Space\\College\\6th_SEM\\Deep_Learning_Project\\IDS\\models\\label_encoder.pkl']

<h2>STEP 5 – Separate Features and Target</h2>

In [29]:
X = data.drop(['Label', 'Attack_Category', 'Encoded_Label'], axis=1)
y = data['Encoded_Label']

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (2520751, 78)
y shape: (2520751,)


<h2>STEP 6 - Feature Scaling</h2>

In [35]:
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib

# Convert to float32 (reduces memory by 50%)
X = X.astype(np.float32)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Save scaler
joblib.dump(scaler, r"D:\My_Space\College\6th_SEM\Deep_Learning_Project\IDS\models\scaler.pkl")

print("Scaling complete.")

Scaling complete.


<h2>STEP 7 - Train/Validation/Split</h2>

In [32]:
from sklearn.model_selection import train_test_split

# First split: Train vs Temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_scaled, y, test_size=0.3, stratify=y, random_state=42
)

# Split temp into validation & test (15% each)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

Train shape: (1764525, 78)
Validation shape: (378113, 78)
Test shape: (378113, 78)


<h2>STEP 8 - Compute Class Weights </h2>

In [33]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights_dict = dict(enumerate(class_weights))
print(class_weights_dict)

{0: np.float64(0.17188427992709365), 1: np.float64(184.80571847507332), 2: np.float64(39.3559718969555), 3: np.float64(2.8130231001004353), 4: np.float64(1.8586723295064924), 5: np.float64(3.9705604385218787), 6: np.float64(168.05)}


<h2>STEP 7 – Reshape for RNN / BiLSTM</h2>

In [34]:
# Reshape to (samples, timesteps, 1)
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_val = X_val.reshape(X_val.shape[0], X_val.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print("Reshaped for RNN:")
print(X_train.shape)

Reshaped for RNN:
(1764525, 78, 1)
